# Feature Engineering
Create a Dataframe for model input:
- Feature creation
- Daily aggregation of features
- Vizualization

# Import Libraries and Data

In [1]:
# --- Standard Libraries ---
import os
import re
from collections import Counter

# --- Data Science ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# --- NLP ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory, LangDetectException

# --- Transformers ---
import torch
from torch.nn.functional import sigmoid
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForSequenceClassification
)
from scipy.special import softmax

# --- Utils ---
from tqdm.auto import tqdm
from IPython.display import display

# --- Setup ---
tqdm.pandas()
nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\paull\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\paull\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Kontrollvariablen

In [ ]:
# Uses shorter time periods for testing purposes
TEST = False

# Wenn nur einzelne Spalten/Features hinzugefügt werden sollen, bitte unten den Block 'neue Features zur CSV hinzufügen' entsprechend anpassen. Die bestehende final_daily_df csv wird dann in ein df geladen und die neuen Spalten werden dazugemerged und die csv wieder abgespeichert.
einzelne_features_zur_bestehenden_CSV_hinzufügen = False # default = False

# wenn True, wird die final_daily_df CSV ganz neu zusammengestellt.
vollstaendige_neuerstellung_der_csv = True # default = False

In [3]:
musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["isRetweet"] = df["isRetweet"].astype(str).str.lower()
    df["possiblySensitive"] = df["possiblySensitive"].astype(str).str.lower()
    df["fullText"] = df["fullText"].astype(str)

musk_twitter_data_nlp["text_raw"] = musk_twitter_data_nlp["text_raw"].astype(str)
musk_twitter_data_nlp["text_lemmatized"] = musk_twitter_data_nlp["text_lemmatized"].astype(str)

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["date"] = df["createdAt"].dt.date

if TEST:
    start_date = pd.to_datetime("2025-04-01").date()
else:
    start_date = pd.to_datetime("2015-01-01").date()

end_date = musk_twitter_data_all["date"].max()

mask_all = (musk_twitter_data_all["date"] >= start_date) & (musk_twitter_data_all["date"] <= end_date)
musk_twitter_data_all = musk_twitter_data_all.loc[mask_all].reset_index(drop=True)

mask_nlp = (musk_twitter_data_nlp["date"] >= start_date) & (musk_twitter_data_nlp["date"] <= end_date)
musk_twitter_data_nlp = musk_twitter_data_nlp.loc[mask_nlp].reset_index(drop=True)

final_daily_df_base = pd.DataFrame({
    'date': pd.date_range(start=start_date, end=end_date)
})
final_daily_df_base["date"] = final_daily_df_base["date"].dt.date 

print("NLPTweets:", musk_twitter_data_nlp.shape, "AllTweets:", musk_twitter_data_all.shape)
musk_twitter_data_nlp.info()
musk_twitter_data_all.info()

C:\Users\paull\AppData\Local\Temp\ipykernel_21484\3089337255.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
C:\Users\paull\AppData\Local\Temp\ipykernel_21484\3089337255.py:2: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])


NLPTweets: (68, 29) AllTweets: (404, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   id                        68 non-null     int64              
 1   url                       68 non-null     object             
 2   twitterUrl                68 non-null     object             
 3   fullText                  68 non-null     object             
 4   retweetCount              68 non-null     float64            
 5   replyCount                68 non-null     float64            
 6   likeCount                 68 non-null     float64            
 7   quoteCount                68 non-null     float64            
 8   viewCount                 68 non-null     float64            
 9   createdAt                 68 non-null     datetime64[ns, UTC]
 10  bookmarkCount             68 non-null     float

# Tweet activity
New features:
- Number of tweets per day

In [4]:
tweet_counts_daily = (
    musk_twitter_data_all
    .groupby("date")
    .size()
    .reset_index(name="tweet_count")
)

# Engagement metrics
- like_count
- quoted_count
- retweet_count
- view_count

In [5]:
# Engagement metrics calculation: Like, Quote, Retweet, Reply counts per day
engagement_metrics = (
    musk_twitter_data_all
    .groupby('date')[['likeCount', 'quoteCount', 'retweetCount', 'replyCount']]
    .sum()
    .astype(int)
    .reset_index()
)


# Sentiment Analysis

New Features:
- Poitve, Neutral ans Negative percentage of posts
- Polarization: Tweets with pos/neg > 0,6

In [6]:
# Tweet sentiment analysis
# Model: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def preprocess(text):
    return text.replace("\n", " ").strip()

def get_sentiment_probs(text):
    text = preprocess(text)
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.cpu().numpy()[0])
    return {
        "sentiment": ['negative', 'neutral', 'positive'][probs.argmax()],
        "neg": probs[0],
        "neu": probs[1],
        "pos": probs[2],
    }

def polarized_label(row):
    return "polarized" if max(row["pos"], row["neg"]) > 0.6 else "not_polarized"

results = musk_twitter_data_nlp["text_raw"].progress_apply(get_sentiment_probs).apply(pd.Series)
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp, results], axis=1)
max_sent = musk_twitter_data_nlp[["pos", "neg"]].max(axis=1)
musk_twitter_data_nlp["sentiment_polarity"] = np.where(max_sent > 0.6, "polarized", "not_polarized")

# 1) Unweighted daily aggregation (mean) with only the “polarized” share
sentiment_avg = (musk_twitter_data_nlp.groupby("date")[["neg", "neu", "pos"]].mean().reset_index())
nlp_counts = (musk_twitter_data_nlp.groupby("date").size().reset_index(name="nlp_tweet_count"))

polar_mean = (musk_twitter_data_nlp.groupby("date")["sentiment_polarity"].apply(lambda s: (s == "polarized").mean()).reset_index(name="polarized"))

sentiment_daily = (
    sentiment_avg
    .merge(nlp_counts, on="date", how="left")
    .merge(polar_mean,  on="date", how="left")
)


# 2) Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(
        neg_w = lambda df: df["neg"] * df["engagement_index"],
        neu_w = lambda df: df["neu"] * df["engagement_index"],
        pos_w = lambda df: df["pos"] * df["engagement_index"],
    )
    .groupby("date")
    .agg(
        neg_w_sum        = ("neg_w", "sum"),
        neu_w_sum        = ("neu_w", "sum"),
        pos_w_sum        = ("pos_w", "sum"),
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
    .assign(
        neg = lambda df: df["neg_w_sum"] / df["total_engagement"],
        neu = lambda df: df["neu_w_sum"] / df["total_engagement"],
        pos = lambda df: df["pos_w_sum"] / df["total_engagement"],
    )
    .drop(columns=["neg_w_sum", "neu_w_sum", "pos_w_sum"])
)

# 2b) Weighted polarization (only “polarized”)
polar_weighted = (
    musk_twitter_data_nlp
    .groupby(["date", "sentiment_polarity"])["engagement_index"]
    .sum()
    .reset_index(name="eng_w_sum")
    .pivot(index="date", columns="sentiment_polarity", values="eng_w_sum")
    .fillna(0)
    .reset_index()
    .merge(weighted_sums[["date", "total_engagement"]], on="date", how="left")
    .assign(polarized=lambda df: df["polarized"] / df["total_engagement"])
    [["date", "polarized"]]
)

# 2c) Final weighted daily DataFrame
sentiment_daily_weighted = (
    weighted_sums[["date", "neg", "neu", "pos"]]
    .merge(polar_weighted, on="date", how="left")
    .merge(nlp_counts,        on="date", how="left")
)


  0%|          | 0/68 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


# Emotions & Personality

New Features:
- Ekman Emotions: anger, disgust, fear, joy, neutral, sadness, surprise
- Big 5 personality traits: Extroversion, Neuroticism, Agreeableness, Conscientiousness, Openness

In [7]:
# Ekman Emotionen
# Model: https://huggingface.co/j-hartmann/emotion-english-distilroberta-base
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Helper that returns a dict of probabilities
def get_emotions(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        logits = model(**tokens).logits
    probs = softmax(logits.numpy()[0])
    return dict(zip(emotion_labels, probs))

# Apply to every tweet
print("Calculating emotion probabilities...")
emotion_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_emotions).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), emotion_probs],axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily emotions...")
emotion_daily = (musk_twitter_data_nlp.groupby('date')[emotion_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily emotions (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{emo}_w": lambda df, emo=emo: df[emo] * df["engagement_index"]
        for emo in emotion_labels
    })
    .groupby("date")
    .agg(
        **{f"{emo}_w_sum": (f"{emo}_w", "sum") for emo in emotion_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

emotion_daily_weighted = (
    weighted_sums
    .assign(**{
        emo: lambda df, emo=emo: df[f"{emo}_w_sum"] / df["total_engagement"]
        for emo in emotion_labels
    })
    [["date", *emotion_labels]]
)
print("Done!")

Calculating emotion probabilities...


  0%|          | 0/68 [00:00<?, ?it/s]

Aggregating daily emotions...
Aggregating daily emotions (weighted)...
Done!


In [8]:
# Big Five Personality Traits
# Model: https://huggingface.co/Minej/bert-base-personality
tokenizer = BertTokenizer.from_pretrained("Minej/bert-base-personality")
model = BertForSequenceClassification.from_pretrained("Minej/bert-base-personality")

personality_labels = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']

def get_personality(text):
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = sigmoid(outputs.logits).squeeze().numpy()
    return dict(zip(personality_labels, probs))

# Apply to every tweet
print("Calculating personality traits...")
personality_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_personality).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), personality_probs], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily personality...")
personality_daily = (musk_twitter_data_nlp.groupby('date')[personality_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily personality (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{pers}_w": lambda df, pers=pers: df[pers] * df["engagement_index"]
        for pers in personality_labels
    })
    .groupby("date")
    .agg(
        **{f"{pers}_w_sum": (f"{pers}_w", "sum") for pers in personality_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

personality_daily_weighted = (
    weighted_sums
    .assign(**{
        pers: lambda df, pers=pers: df[f"{pers}_w_sum"] / df["total_engagement"]
        for pers in personality_labels
    })
    [["date", *personality_labels]]
)
print("Done!")

Calculating personality traits...


  0%|          | 0/68 [00:00<?, ?it/s]

Aggregating daily personality...
Aggregating daily personality (weighted)...
Done!


# Topic and word counts

New Features: 
- Daily Word counts
    - Rationale of Definition of words:
        - Company/ticker terms (e.g. tesla, tsla, spacex) capture direct references to publicly traded entities.
        - Product names (e.g. model, cybertruck, starship) often precede news that can move stock prices.
        - Crypto tokens (e.g. bitcoin, dogecoin, ethereum, crypto) map to Musk-driven volatility in the digital-asset markets
        - Financial keywords (e.g. stock, market, price, profit, loss, revenue) directly signal earnings or valuation discussions.
        - Macro terms (e.g. inflation, interest) reflect broader economic commentary that can sway sentiment.
        - Action verbs (buy, sell) often presage trading intent or recommendations.
- Topics of posts

In [9]:
# Words
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\S+|[^a-z\s]", "", text)
    return text.split()

all_tokens = musk_twitter_data_nlp["text_lemmatized"].dropna().apply(tokenize)
flat_tokens = [token for sublist in all_tokens for token in sublist]
word_counts = Counter(flat_tokens)
word_counts = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

top20 = [
    'tesla', 'stock', 'market', 'price', 'profit', 'loss', 'revenue',
    'inflation', 'interest', 'bitcoin', 'dogecoin', 'crypto', 'ethereum',
    'spacex', 'model', 'cybertruck', 'starship', 'buy', 'sell'
]

top_word_df = musk_twitter_data_nlp.dropna(subset=['text_lemmatized']).copy()
top_word_df['tokens'] = top_word_df['text_lemmatized'].apply(tokenize)
top_word_df = top_word_df.explode('tokens')
top_word_df['tokens'] = top_word_df['tokens'].replace({'tsla': 'tesla'})

top_word_df = top_word_df[top_word_df['tokens'].isin(top20)].copy()

daily_word_counts = (
    top_word_df
    .groupby(['date','tokens'])
    .size()
    .unstack(fill_value=0)
)

daily_word_counts = daily_word_counts.reindex(
    columns=top20,
    fill_value=0
).sort_index()

In [10]:
# Topic Analysis
# Model: https://huggingface.co/cardiffnlp/tweet-topic-21-multi
model_name = "cardiffnlp/tweet-topic-21-multi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

topic_labels = [
    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",
    "diaries_daily_life", "family", "fashion_style", "film_tv_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",
    "music", "news_social_concern", "other_hobbies", "relationships",
    "science_technology", "sports", "travel_adventure", "youth_student_life"
]

def get_topics(text):
    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.numpy()[0])
    return dict(zip(topic_labels, probs))

# Apply to every tweet
print("Calculating topic probabilities...")
topic_scores = musk_twitter_data_nlp['text_lemmatized'].progress_apply(get_topics).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), topic_scores], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily topics...")
topics_daily = (musk_twitter_data_nlp.groupby('date')[topic_labels].mean().reset_index())

# Weighted daily aggregation
print("Aggregating daily topics (weighted)...")
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{top}_w": lambda df, top=top: df[top] * df["engagement_index"]
        for top in topic_labels
    })
    .groupby("date")
    .agg(
        **{f"{top}_w_sum": (f"{top}_w", "sum") for top in topic_labels},
        total_engagement=("engagement_index", "sum"),
    )
    .reset_index()
)

# 2b) Traits wieder auf die Originalnamen zurückskalieren
topics_daily_weighted = (
    weighted_sums
    .assign(**{
        top: lambda df, top=top: df[f"{top}_w_sum"] / df["total_engagement"]
        for top in topic_labels
    })
    [["date", *topic_labels]]
)
print("Done!")

Calculating topic probabilities...


  0%|          | 0/68 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Aggregating daily topics...
Aggregating daily topics (weighted)...
Done!


In [11]:
display(sentiment_daily_weighted.head())
display(emotion_daily_weighted.head())
display(emotion_daily.head())

display(personality_daily_weighted.head())
display(personality_daily.head())
display(daily_word_counts.head())
display(topics_daily_weighted.head())


,date,neg,neu,pos,polarized,nlp_tweet_count
0,2025-04-01,0.255971,0.523556,0.220473,0.471972,10
1,2025-04-02,0.220453,0.498079,0.281468,0.357037,9
2,2025-04-03,0.230607,0.224880,0.544513,0.790851,5
3,2025-04-04,0.353538,0.411695,0.234767,0.498417,9
4,2025-04-05,0.115573,0.714937,0.169490,0.000000,3


,date,anger,disgust,fear,joy,neutral,sadness,surprise
0,2025-04-01,0.023949,0.024675,0.022463,0.122194,0.546545,0.017016,0.243159
1,2025-04-02,0.036563,0.106797,0.096333,0.062984,0.570848,0.010327,0.116148
2,2025-04-03,0.240208,0.017625,0.030317,0.191660,0.422783,0.018423,0.078984
3,2025-04-04,0.094505,0.184978,0.020470,0.109326,0.480775,0.028503,0.081444
4,2025-04-05,0.017401,0.007318,0.025707,0.007506,0.764941,0.020230,0.156896


,date,anger,disgust,fear,joy,neutral,sadness,surprise
0,2025-04-01,0.029571,0.026590,0.025089,0.169525,0.464212,0.027018,0.257995
1,2025-04-02,0.034038,0.095088,0.141109,0.048335,0.561679,0.010044,0.109706
2,2025-04-03,0.216779,0.014374,0.048943,0.206525,0.403062,0.025866,0.084451
3,2025-04-04,0.065097,0.144178,0.016518,0.136424,0.520599,0.024418,0.092767
4,2025-04-05,0.015982,0.007270,0.022595,0.008979,0.775972,0.023212,0.145990


,date,Extroversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,2025-04-01,0.499008,0.554966,0.433459,0.288884,0.503211
1,2025-04-02,0.466013,0.546644,0.447591,0.302712,0.493875
2,2025-04-03,0.539179,0.571418,0.404561,0.257888,0.544117
3,2025-04-04,0.471063,0.553703,0.450331,0.313059,0.494804
4,2025-04-05,0.322956,0.553815,0.453653,0.298440,0.419273


,date,Extroversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,2025-04-01,0.498899,0.552688,0.439282,0.293181,0.502147
1,2025-04-02,0.456464,0.542893,0.454616,0.312197,0.481922
2,2025-04-03,0.523928,0.568547,0.411923,0.259827,0.531875
3,2025-04-04,0.467840,0.557798,0.444472,0.302613,0.491506
4,2025-04-05,0.333637,0.551909,0.444723,0.291843,0.429999


tokens,tesla,stock,market,price,profit,loss,revenue,inflation,interest,bitcoin,dogecoin,crypto,ethereum,spacex,model,cybertruck,starship,buy,sell
date,,,,,,,,,,,,,,,,,,,
2025-04-02,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2025-04-03,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0
2025-04-10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2025-04-11,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


,date,arts_culture,business_entrepreneurs,celebrity_pop_culture,diaries_daily_life,family,fashion_style,film_tv_video,fitness_&_health,food_&_dining,gaming,learning_educational,music,news_social_concern,other_hobbies,relationships,science_technology,sports,travel_adventure,youth_student_life
0,2025-04-01,0.005028,0.053037,0.005535,0.057552,0.001161,0.001149,0.032044,0.000865,0.001125,0.001906,0.009668,0.002609,0.513292,0.011548,0.001578,0.265318,0.004738,0.029649,0.002197
1,2025-04-02,0.010479,0.018284,0.012657,0.195192,0.002093,0.050912,0.009575,0.005309,0.001841,0.004034,0.020713,0.005428,0.287294,0.036699,0.004656,0.311497,0.009117,0.009061,0.005161
2,2025-04-03,0.016603,0.018499,0.012134,0.173253,0.002899,0.074744,0.011192,0.001545,0.002365,0.009821,0.026241,0.002640,0.353088,0.098859,0.005483,0.081949,0.096057,0.008966,0.003661
3,2025-04-04,0.041801,0.008180,0.007910,0.395716,0.003710,0.001357,0.015016,0.001186,0.002729,0.004082,0.015027,0.018970,0.219630,0.110962,0.007926,0.036750,0.087151,0.017886,0.004009
4,2025-04-05,0.058376,0.009758,0.071582,0.240048,0.013116,0.057586,0.055511,0.003559,0.006846,0.003168,0.008464,0.018372,0.354575,0.050135,0.014137,0.004307,0.017376,0.009538,0.003546


## Additional Features to consider/ ToDos

- ToDo Tweet-Typ (z. B. Meme, Information, Ankündigung, Meinung, Engagement)
    - Studien zeigen, dass z. B. Meme-Posts und ironische Tweets besonders starke Kursreaktionen auslösen 
    - Bei Musk besonders relevant, da sein Kommunikationsstil sich im Zeitverlauf stark verändert hat 


Aus Termin mit Peter:
- Einflussreiche weitere Personen: Kann man ggf auch aus quotes nehmen, ist mir nicht mehr ganz klar was er wollte.
- Quotes mit einbeziehen, Quote dataset enthält die texte der Quotes -> Einbeziehen, höhere Genauigkeit bei eg toics

# Create final Daily DF
One can just add Features to the existing Dataframe or create a new Final Daily Df
### Add new Features to existing dataframe

In [12]:
if einzelne_features_zur_bestehenden_CSV_hinzufügen:
    # 1. Final-Dataset laden mit geparster Datumsspalte
    final_daily_df = pd.read_csv("Data/twitter_data/processed/final_daily_df.csv", parse_dates=["date"])

    # 2. Platzhaltervariable für zusätzliche Feature-DataFrames
    # Beispiel: zusatz_feature_dfs = [df_neues_feature_1, df_neues_feature_2, ...]
    zusatz_feature_dfs = [
        
        # HIER DIE OBEN ERSTELLTEN NEUEN SPALTEN (inkl. 'date' spalte) AUFLISTEN
        engagement_metrics
    ]

    # 3. Iterativ mergen
    for feature_df in zusatz_feature_dfs:
        
        feature_df["date"] = pd.to_datetime(feature_df["date"])
        
        # Prüfen auf doppelte Spalten (außer 'date')
        doppelte = [col for col in feature_df.columns if col != "date" and col in final_daily_df.columns]
        if doppelte:
            raise ValueError(f"Die folgenden Spalten sind bereits in final_daily_df vorhanden und sollten evtl. nicht erneut gemerged werden: {doppelte}")

        # Merge auf 'date'
        final_daily_df = pd.merge(final_daily_df, feature_df, on="date", how="left")

    # 4. Ergebnis zurückschreiben
    final_daily_df.to_csv("Data/twitter_data/processed/final_daily_df.csv", index=False)


### Merge and Create final df
Merge the daily dfs in one new dataframe and create csv (creates a weighted and an unweighted version)

In [13]:
# Merge with complete date, fill missing days with zero
if vollstaendige_neuerstellung_der_csv:
    # Unweighted final daily DataFrame
    final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    final_daily_df = final_daily_df.merge(engagement_metrics, on="date", how="left")
    final_daily_df["tweet_count"] = final_daily_df["tweet_count"].astype(int)
    final_daily_df = final_daily_df.merge(sentiment_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(emotion_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(personality_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(daily_word_counts, on="date", how="left")
    final_daily_df = final_daily_df.merge(topics_daily, on="date", how="left")
    final_daily_df["no_tweets"] = (final_daily_df["tweet_count"] == 0).astype(int)
    display(final_daily_df.info())
    display(final_daily_df.head())
    # Weighted final daily DataFrame
    weighted_final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    weighted_final_daily_df = weighted_final_daily_df.merge(engagement_metrics, on="date", how="left")
    weighted_final_daily_df["tweet_count"] = weighted_final_daily_df["tweet_count"].astype(int)
    weighted_final_daily_df = weighted_final_daily_df.merge(sentiment_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(emotion_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(personality_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(daily_word_counts, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(topics_daily_weighted, on="date", how="left")
    weighted_final_daily_df["no_tweets"] = (weighted_final_daily_df["tweet_count"] == 0).astype(int)

    display(weighted_final_daily_df.info())
    display(weighted_final_daily_df.head())

### Export

In [14]:
if vollstaendige_neuerstellung_der_csv:
    final_daily_df.to_csv(os.path.join('processed', 'final_daily_df.csv'), index=False)
    weighted_final_daily_df.to_csv(os.path.join('processed', 'weighted_final_daily_df.csv'), index=False)